In [1]:

# PASO 1: Abrimos librerias necesarias
import json
import pandas as pd

In [2]:
import json

archivos = [
    "Files/api_artistas_hip_hop.json", 
    "Files/datos_latin_apilastfm.json", 
    "Files/datos_pop_apilastfm.json", 
    "Files/datos_rap_apilastfm.json",
    "Files/datos_rock_apilastfm.json"
]

lista_unificada = []

for nombre in archivos:
    with open(nombre, "r", encoding="utf-8") as f:
        datos = json.load(f)
        
        for artista in datos:
            # --- AQUÍ ESTÁ EL TRUCO ---
            # Si existe "biografía" (con tilde), la movemos a "biografia" (sin tilde)
            if "biografía" in artista:
                artista["biografia"] = artista.pop("biografía")
            
            lista_unificada.append(artista)



In [3]:
len(lista_unificada)

6057

In [4]:
from collections import Counter
nombres = [str(a.get("artista")).strip().lower() for a in lista_unificada]
duplicados = [item for item, count in Counter(nombres).items() if count > 1]
print(f"Artistas que Python detecta como duplicados: {duplicados}")

Artistas que Python detecta como duplicados: ['xxxtentacion', 'the alchemist', 'natos y waor', 'arcángel', 'bad bunny', 'losniñosdelcaminito', 'khea', 'duki', 'cazzu', 'becky g', 'prince royce', 'dj luian', 'mambo kingz', 'anuel aa', 'c. tangana', 'kanye west', 'kidd keo', 'delaossa', 'j.moods', 'trueno', 'blake', 'j-hope', 'soolking', 'matasvandals', 'da silva', 'rels b', 'diplo', 'fernandocosta', 'chicho beats', 'denom', 'nadal015', 'kaze', 'el perla', 'drake', 'fazzini', 'franky style', 'moha la squale', 'miranda', 'wos', 'javi bambini cattivi', 'alemán', 'revol', 'j balvin', 'de la ghetto', 'dirty porko', 'drefquila', 'lil pump', 'swae lee', 'maluma', 'trippie redd', 'eladio carrion', 'big soto', 'ysy a', 'hard gz', 'eminem', 'mac miller', 'lil gz', 'calero ldn', 'trap capos', 'noriel', 'bryant myers', 'walls', 'el jincho', 'sfdk', 'cardi b', 'lil baby', 'gunna', 'costa', 'ninho', 'residente', 'nicky jam', 'a boogie wit da hoodie', 'post malone', 'nanpa básico', 'toteking', 'bizarr

In [5]:
len(duplicados)

1062

#paso muy importante para poder evitar duplicados y poner artista como PK

In [6]:
datos_artista_para_sql_api = []
# Usamos un set para nombres normalizados
artistas_vistos = set()

for artista in lista_unificada:
    nombre_original = artista.get("artista")
    
    if nombre_original is None:
        continue

    # --- NORMALIZACIÓN RADICAL ---
    # Quitamos espacios, saltos de línea y pasamos a minúsculas
    nombre_limpio = str(nombre_original).strip().lower()
    
    # Si ya vimos este nombre (aunque tuviera espacios o mayúsculas distintas), lo saltamos
    if nombre_limpio in artistas_vistos:
        continue
    
    # Marcamos como visto
    artistas_vistos.add(nombre_limpio)
    
    # --- PROCESAMIENTO ---
    fila = artista.copy()
    
    # Limpiamos también el nombre que irá a SQL para que no haya espacios
    fila["artista"] = nombre_original.strip()
    
    if isinstance(fila.get("similares"), list):
        fila["similares"] = ", ".join(fila["similares"])
    
    if fila.get("similares") is None:
        fila["similares"] = ""
        
    datos_artista_para_sql_api.append(fila)

print(f"Total original: {len(lista_unificada)}")
print(f"Total tras limpieza radical: {len(datos_artista_para_sql_api)}")

Total original: 6057
Total tras limpieza radical: 3268


In [6]:
len(artistas_vistos)

3268

In [ ]:
# Ahora guardamos todo limpio
with open("Files/json_general_limpio.json", "w", encoding="utf-8") as f:
    json.dump(lista_unificada, f, indent=4, ensure_ascii=False)

print("¡Fusionado y normalizado! Ahora solo tienes una clave 'biografia'.")

In [4]:
df_lista_unificada = pd.DataFrame(lista_unificada)

In [5]:
df_lista_unificada

,artista,audiencia,reproducciones,similares,biografia
0,XXXTENTACION,2774772,335541691,Kid Trunks,"Jahseh Dwayne Onfroy (January 23, 1998 – June ..."
1,The Alchemist,1347530,47088057,Roc Marciano,"Alan Daniel Maman (born October 25, 1977), bet..."
2,Action Bronson,807075,25853544,Action Bronson & The Alchemist,"Ariyan Arslani (born December 2, 1983), better..."
3,Natos y Waor,52124,2763301,FERNANDOCOSTA,Dúo de rap madrileño. Cuentan con 8 trabajos a...
4,Arcángel,433430,14000648,De La Ghetto,"Austin Santos, conocido como Arcangel, nació e..."
...,...,...,...,...,...
6052,Совче,106,341,"[SH Kera, Весъ, Plastic Robots, TRUEten, Брату...",
6053,Стриж,8461,167517,"[Дымовая Завеса, Птаха, 5 Плюх, TAHDEM Foundat...",
6054,ТРИДЦАТЬ ДЕВЯТЫЙ,1439,15141,"[JEEMBO & TVETH, SUPERIOR.CAT.PROTEUS, Pkhat, ...",
6055,ЦИНК УРОДОВ,10025,183850,"[тяжёлая атлетика, H8.HOOD, Metox, Алкоголь По...",


In [ ]:

with open("Files/json_general_limpio.json.json", "w", encoding='utf-8') as f: 
    json.dump(lista_unificada, f, indent=4, ensure_ascii=False)

In [ ]:
#pip install SQLAlchemy

In [7]:
import mysql.connector
from mysql.connector import errorcode
import pandas as pd
import json
from sqlalchemy import create_engine

In [ ]:
usuario = "root"
password = "UsuarioAdalab"
host = "127.0.0.1"
base_de_datos = "pandemusic" # Asegúrate de haber creado el Schema en Workbench

In [9]:
#creamos el engine a ver que tal
engine = create_engine(f"mysql+mysqlconnector://{usuario}:{password}@{host}/{base_de_datos}")
try:
    df_lista_unificada.to_sql('tabla_artistas', con=engine, if_exists="replace", index=False)
    print("✅ ¡Conexión exitosa! Los artistas se han guardado en la tabla 'artistas'.")
except Exception as e:
    print(f"❌ Ups, algo falló: {e}")

❌ Ups, algo falló: Execution failed on sql 'INSERT INTO tabla_artistas (artista, audiencia, reproducciones, similares, biografia) VALUES (:artista, :audiencia, :reproducciones, :similares, :biografia)': (mysql.connector.errors.InterfaceError) Failed executing the operation; Python type list cannot be converted
[SQL: INSERT INTO tabla_artistas (artista, audiencia, reproducciones, similares, biografia) VALUES (%(artista)s, %(audiencia)s, %(reproducciones)s, %(similares)s, %(biografia)s)]
[parameters: [{'artista': 'XXXTENTACION', 'audiencia': '2774772', 'reproducciones': '335541691', 'similares': 'Kid Trunks', 'biografia': 'Jahseh Dwayne Onfroy (January 23, 1998 – June 18, 2018), better known by his stage name XXXTentacion, was an American rapper from Lauderhill, Florida ... (273 characters truncated) ... nd unmastered. His vocal style ranged from singing, rapping, and screaming. <a href="https://www.last.fm/music/XXXTENTACION">Read more on Last.fm</a>'}, {'artista': 'The Alchemist', 'aud

In [13]:
datos_para_sql_api = []

for artista in lista_unificada:
    # Creamos una copia para no romper el original
    fila = artista.copy()
    
    # --- EL TRUCO ESTÁ AQUÍ ---
    # Si 'similares' es una lista, la convertimos a un string separado por comas
    if isinstance(fila.get("similares"), list):
        fila["similares"] = ", ".join(fila["similares"])
    
    # Si es None o no existe, le ponemos un texto vacío para que SQL no sufra
    if fila.get("similares") is None:
        fila["similares"] = ""
        
    datos_para_sql_api.append(fila)

# Ahora ya puedes lanzar tu SQL con 'datos_para_sql'
# engine.execute(sql, datos_para_sql) o el método que estés usando

In [14]:
df_datos_para_sql_api = pd.DataFrame(datos_para_sql_api)

In [15]:
df_datos_para_sql_api

,artista,audiencia,reproducciones,similares,biografia
0,XXXTENTACION,2774772,335541691,Kid Trunks,"Jahseh Dwayne Onfroy (January 23, 1998 – June ..."
1,The Alchemist,1347530,47088057,Roc Marciano,"Alan Daniel Maman (born October 25, 1977), bet..."
2,Action Bronson,807075,25853544,Action Bronson & The Alchemist,"Ariyan Arslani (born December 2, 1983), better..."
3,Natos y Waor,52124,2763301,FERNANDOCOSTA,Dúo de rap madrileño. Cuentan con 8 trabajos a...
4,Arcángel,433430,14000648,De La Ghetto,"Austin Santos, conocido como Arcangel, nació e..."
...,...,...,...,...,...
6052,Совче,106,341,"SH Kera, Весъ, Plastic Robots, TRUEten, Братубрат",
6053,Стриж,8461,167517,"Дымовая Завеса, Птаха, 5 Плюх, TAHDEM Foundati...",
6054,ТРИДЦАТЬ ДЕВЯТЫЙ,1439,15141,"JEEMBO & TVETH, SUPERIOR.CAT.PROTEUS, Pkhat, Ж...",
6055,ЦИНК УРОДОВ,10025,183850,"тяжёлая атлетика, H8.HOOD, Metox, Алкоголь Пос...",


In [16]:
#creamos el engine a ver que tal
engine = create_engine(f"mysql+mysqlconnector://{usuario}:{password}@{host}/{base_de_datos}")
try:
    df_datos_para_sql_api.to_sql('tabla_artistas', con=engine, if_exists="replace", index=False)
    print("✅ ¡Conexión exitosa! Los artistas se han guardado en la tabla 'artistas'.")
except Exception as e:
    print(f"❌ Ups, algo falló: {e}")

✅ ¡Conexión exitosa! Los artistas se han guardado en la tabla 'artistas'.
